# Big Data  y Machine Learning (UBA) 
## Clase 6 - Modelo de regresión lineal, multicolinealidad y error de pronostico

**Objetivo:**  
Que se familiaricen con los problemas de multicolinealidad, con su efecto en el error cuadrático medio ($ECM(\hat{f})$) del modelo y el error de pronóstico ($Err(\hat{Y})$)

### Temario:
- Familiarización con la base de salarios de baseballistas (*hitters*)
- Ejercicio de PCA con y sin salario: Implicancia económica del salario
- Familiarización con muestra de entrenamiento y de testeo
- Predicción de salarios con distintas especificaciones
- Comparación de desempeño de modelos cuando cambiamos los datos


### 1. Familiarización con la base de salarios de baseballistas

Exploraremos brevemente el conjunto de datos Baseballistas (["Hitters"](https://islp.readthedocs.io/en/latest/datasets/Hitters.html)) y usaremos la librería de sklearn para hacer un ejercicio no supervisado y supervisado. Primero, con histogramas y el análisis de componentes principales visualizaremos patrones en los datos. Segundo, con la regresión lineal, haremos el ejercicio supervisado de predecir el salario de los jugadores de béisbol.

##### Baseball data, 'Hitters'
Datos de la Major League de Baseball Data en las temporadas 1986 y 1987.
La base de Hitters tiene las siguientes variables:
- AtBat: Number of times at bat in 1986
- Hits: Number of hits in 1986
- HmRun: Number of home runs in 1986
- Runs: Number of runs in 1986
- RBI: Number of runs batted in in 1986
- Walks: Number of walks in 1986
- Years: Number of years in the major leagues
- CAtBat: Number of times at bat during his career
- CHits: Number of hits during his career
- CHmRun: Number of home runs during his career
- CRuns: Number of runs during his career
- CRBI: Number of runs batted in during his career
- CWalks: Number of walks during his career
- League: A factor with levels A and N indicating player’s league at the end of 1986
- Division: A factor with levels E and W indicating player’s division at the end of 1986
- PutOuts: Number of put outs in 1986
- Assists: Number of assists in 1986
- Errors: Number of errors in 1986
- Salary: 1987 annual salary on opening day in thousands of dollars
- NewLeague: A factor with levels A and N indicating player’s league at the beginning of 1987


Nuestro objetivo será **predecir el salario** (regresión)

#### 1.1. Leer el conjunto de datos y explorar la estructura de datos 

In [ ]:
# Importamos los paquetes necesarios
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ISLP import load_data

from sklearn.preprocessing import scale
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
Hitters = load_data('Hitters')
print(Hitters.info())
print('Dimensión de la base:', Hitters.shape, '\n')
#Vemos los missing values en Y
print('\nMissings en variable dependiente:', np.isnan(Hitters['Salary']).sum())

Dado que hay 59 observaciones sin el salario ($Y$ a predecir), eliminamos dichas filas sin variable dependiente.

In [ ]:
Hitters_clean = Hitters.dropna(subset=['Salary']).reset_index(drop=True)
print('\n Nueva dimensión de la base:', Hitters_clean.shape)

In [ ]:
Hitters_clean

#### 1.2. Preparar las X e Y que usaremos en el modelo

Aquí seleccionamos las variables que utilizaremos en nuestro modelo y transformamos a dummies las que son strings

In [ ]:
y = Hitters_clean.Salary
y.shape

Rapidamente, veamos la distribucion de salarios de los baseballistas

In [ ]:
y.describe().round(2)

##### 1.2 Visualización de los Salarios
Creamos una linda distribucion de los salarios siguiendo nuestros principios de visualización de la clase 1.

In [ ]:
sns.set_style("white")  # Cambié de "whitegrid" a "white"
plt.rcParams['figure.facecolor'] = 'white'

fig, ax = plt.subplots(figsize=(12, 7))

# KDE plot con histograma de fondo opcional
sns.kdeplot(
    data=Hitters_clean,
    x='Salary',
    fill=True,
    color='#2E86AB',
    alpha=0.5,
    linewidth=1.5,
    ax=ax,
    bw_adjust=0.9  # Ajustar suavidad del kernel
)

# Estadísticas
mediana = Hitters_clean['Salary'].median()
p10 = Hitters_clean['Salary'].quantile(0.10)
p90 = Hitters_clean['Salary'].quantile(0.90)
media = Hitters_clean['Salary'].mean()

# Línea mediana (sólida, roja)
ax.axvline(mediana, color='#E63946', linestyle='-', linewidth=2.5, 
           label=f'Mediana: ${mediana:.0f}K', zorder=3)

# Percentiles (punteadas)
ax.axvline(p10, color='#F77F00', linestyle='--', linewidth=2.2, 
           label=f'P10: ${p10:.0f}K', zorder=2)
ax.axvline(p90, color='#06A77D', linestyle='--', linewidth=2.2, 
           label=f'P90: ${p90:.0f}K', zorder=2)


# Diseño
ax.set_xlabel('Salario (miles de USD)', fontsize=13)
ax.set_ylabel('Densidad', fontsize=13, fontweight='bold')
ax.set_title('Distribución de Salarios de Baseballistas (MLB)\nEstimación de Kernel', 
             fontsize=14, pad=20)

ax.legend(fontsize=11, loc='upper right', framealpha=0.97, edgecolor='gray')

plt.tight_layout()
plt.show()

In [ ]:
Hitters_clean.shape

In [ ]:
print(Hitters_clean.League.value_counts())
print(Hitters_clean.Division.value_counts())
print(Hitters_clean.NewLeague.value_counts())

# Creamos variables dummies para las variables string
dummies = pd.get_dummies(Hitters_clean[['League', 'Division', 'NewLeague']], drop_first=True).reset_index(drop=True)
dummies

In [ ]:
# Eliminar las columnas categóricas originales
Hitters2 = Hitters_clean.drop(['League', 'Division', 'NewLeague'], axis=1).reset_index(drop=True)

## Sumamos las dummies a hitters
Hitters2 = pd.concat([Hitters2, dummies[['League_N', 'Division_W', 'NewLeague_N']].astype(float)], axis=1)

Hitters2.info()

In [ ]:
Hitters2.shape

In [ ]:
Hitters2

In [ ]:
# Definimos las variables que incluiremos en el set de X

# Eliminamos salarios (porque es nuestra y) y las columnas de strings
X_num = Hitters2.drop(['Salary'], axis=1)

X_num.info()

In [ ]:
X_num

**Pregunta:** Cuáles variables muestran alta correlacion lineal?

In [ ]:
# Visualizamos las correlaciones dentro de la base de entrenamiento
colormap = plt.cm.viridis
fig = plt.figure(figsize=(12,12))
plt.title('Correlacion de Pearson entre las Xs', y=1.05, size=15)
mask = np.triu(np.ones_like(Hitters2.astype(float).corr(), dtype=bool), k=1)
sns.heatmap(Hitters2.astype(float).corr(), square=True, cmap=colormap, linecolor='white', annot=True, mask=mask)
fig.savefig("Hitters_corr_X_num.pdf",bbox_inches='tight')


### 2. Ejercicio no supervisado: PCA con y sin salario
Aprovechando la base de datos `Hitters2` con salario y `X_num` sin salario, realicemos el analisis de componentes principales y veamos la importancia de dicha variable para armar los componentes principales.


In [ ]:
# importamos los paquetes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# primero estandarizamos las variables
# Con salario (Hitters2)
scaler1 = StandardScaler()
Hitters2_scaled = scaler1.fit_transform(Hitters2)

# Sin salario (X_num)
scaler2 = StandardScaler()
X_num_scaled = scaler2.fit_transform(X_num)

Chequeamos que la media sea 0 y el desvio 1

In [ ]:
# probar aqui

In [ ]:

# PCA con salario
pca_with_salary = PCA()
scores_with_salary = pca_with_salary.fit_transform(Hitters2_scaled)

# PCA sin salario
pca_without_salary = PCA()
scores_without_salary = pca_without_salary.fit_transform(X_num_scaled)

#### 2.1. Distribucion de los scores con y sin salarios

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Con Salario
axes[0].scatter(scores_with_salary[:, 0], scores_with_salary[:, 1], 
                alpha=0.6, color='#2E86AB', s=50, edgecolors='navy', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca_with_salary.explained_variance_ratio_[0]:.2%})', 
                    fontsize=11, fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca_with_salary.explained_variance_ratio_[1]:.2%})', 
                    fontsize=11, fontweight='bold')
axes[0].set_title('Panel A: PCA CON Salario', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, linestyle=':', linewidth=0.7)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)

# Panel B: Sin Salario
axes[1].scatter(scores_without_salary[:, 0], scores_without_salary[:, 1], 
                alpha=0.6, color='#A23B72', s=50, edgecolors='#6B1B47', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca_without_salary.explained_variance_ratio_[0]:.2%})', 
                    fontsize=11, fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca_without_salary.explained_variance_ratio_[1]:.2%})', 
                    fontsize=11, fontweight='bold')
axes[1].set_title('Panel B: PCA SIN Salario', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, linestyle=':', linewidth=0.7)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

#### 2.2 Loading con y sin salario

In [ ]:
# Loadings = componentes principales * sqrt(varianza explicada)
loadings_with = pca_with_salary.components_.T * np.sqrt(pca_with_salary.explained_variance_)
loadings_without = pca_without_salary.components_.T * np.sqrt(pca_without_salary.explained_variance_)

# Obtener nombres de variables
var_names_with = Hitters2.columns.tolist()
var_names_without = X_num.columns.tolist()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# -------- PANEL A: CON SALARIO --------
ax = axes[0]

# Dibujar flechas desde origen a cada loading
for i, var in enumerate(var_names_with):
    ax.arrow(0, 0, 
             loadings_with[i, 0], loadings_with[i, 1],
             head_width=0.04, head_length=0.04, 
             fc='#2E86AB', ec='navy', alpha=0.7, linewidth=1.5)
    
    # Etiquetar cada variable
    ax.text(loadings_with[i, 0] * 1.15, loadings_with[i, 1] * 1.15,
            var, fontsize=9, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#2E86AB', 
                     alpha=0.2, edgecolor='navy', linewidth=0.5))

# Círculo de correlación (referencia)
circle = plt.Circle((0, 0), 1, color='gray', fill=False, 
                     linestyle='--', linewidth=1, alpha=0.5)
ax.add_patch(circle)

ax.set_xlabel(f'PC1 ({pca_with_salary.explained_variance_ratio_[0]:.2%})', 
              fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca_with_salary.explained_variance_ratio_[1]:.2%})', 
              fontsize=12, fontweight='bold')
ax.set_title('Panel A: Loadings - PCA CON Salario', fontsize=13, fontweight='bold', pad=15)
ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.8, alpha=0.3)
ax.set_aspect('equal')

# Establecer límites
max_load_with = np.max(np.abs(loadings_with[:, :2])) * 1.3
ax.set_xlim([-max_load_with, max_load_with])
ax.set_ylim([-max_load_with, max_load_with])

# -------- PANEL B: SIN SALARIO --------
ax = axes[1]

# Dibujar flechas desde origen a cada loading
for i, var in enumerate(var_names_without):
    ax.arrow(0, 0, 
             loadings_without[i, 0], loadings_without[i, 1],
             head_width=0.04, head_length=0.04, 
             fc='#A23B72', ec='#6B1B47', alpha=0.7, linewidth=1.5)
    
    # Etiquetar cada variable
    ax.text(loadings_without[i, 0] * 1.15, loadings_without[i, 1] * 1.15,
            var, fontsize=9, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#A23B72', 
                     alpha=0.2, edgecolor='#6B1B47', linewidth=0.5))

# Círculo de correlación (referencia)
circle = plt.Circle((0, 0), 1, color='gray', fill=False, 
                     linestyle='--', linewidth=1, alpha=0.5)
ax.add_patch(circle)

ax.set_xlabel(f'PC1 ({pca_without_salary.explained_variance_ratio_[0]:.2%})', 
              fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca_without_salary.explained_variance_ratio_[1]:.2%})', 
              fontsize=12, fontweight='bold')
ax.set_title('Panel B: Loadings - PCA SIN Salario', fontsize=13, fontweight='bold', pad=15)
ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.8, alpha=0.3)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.8, alpha=0.3)
ax.set_aspect('equal')

# Establecer límites
max_load_without = np.max(np.abs(loadings_without[:, :2])) * 1.3
ax.set_xlim([-max_load_without, max_load_without])
ax.set_ylim([-max_load_without, max_load_without])

plt.tight_layout()
plt.show()

#### 2.3 Proporcion de la varianza explicada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Calcular varianza acumulada
var_exp_with = pca_with_salary.explained_variance_ratio_[:10]
var_exp_without = pca_without_salary.explained_variance_ratio_[:10]

cumsum_with = np.cumsum(var_exp_with)
cumsum_without = np.cumsum(var_exp_without)

components = np.arange(1, 11)

# Panel A: Con Salario
axes[0].bar(components, var_exp_with, alpha=0.7, color='#2E86AB', 
            label='Varianza individual', edgecolor='navy', linewidth=1)
axes[0].plot(components, cumsum_with, marker='o', color='#E63946', 
             linewidth=2.5, markersize=6, label='Varianza acumulada')
axes[0].set_xlabel('Componente Principal', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Proporción de Varianza Explicada', fontsize=11, fontweight='bold')
axes[0].set_title('Panel A: Varianza Explicada (CON Salario)', fontsize=12, fontweight='bold')
axes[0].set_xticks(components)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3, linestyle=':', linewidth=0.7, axis='y')
axes[0].set_ylim([0, 1])

# Panel B: Sin Salario
axes[1].bar(components, var_exp_without, alpha=0.7, color='#A23B72', 
            label='Varianza individual', edgecolor='#6B1B47', linewidth=1)
axes[1].plot(components, cumsum_without, marker='o', color='#F77F00', 
             linewidth=2.5, markersize=6, label='Varianza acumulada')
axes[1].set_xlabel('Componente Principal', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Proporción de Varianza Explicada', fontsize=11, fontweight='bold')
axes[1].set_title('Panel B: Varianza Explicada (SIN Salario)', fontsize=12, fontweight='bold')
axes[1].set_xticks(components)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, linestyle=':', linewidth=0.7, axis='y')
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

### 3. Dividimos la base en observaciones para entrenamiento y testeo

Ahora dividimos la muestra en un conjunto de entrenamiento y un conjunto de prueba para luego estimar el error en el conjunto de prueba. 

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X_num_scaled, y, test_size=0.3, random_state=100)

# Revisamos cuantas observaciones quedaron para Test y cuantas para Entrenamiento.
print(f'El conjunto de entrenamiento tiene {len(X_train)} observaciones.')
print(f'El conjunto de test tiene {len(X_test)} observaciones.')

### 4. Modelo de regresión de salarios 
Vamos hacer 5 modelos de regresión lineal, usando 1%, 25%, 50%, 75% y 100% de todos los predictores disponibles.

In [ ]:
especificaciones = {
    'Spec 1: 1% vars': 0.01,
    'Spec 2: 25% vars': 0.25,
    'Spec 3: 50% vars': 0.50,
    'Spec 4: 75% vars': 0.75,
    'Spec 5: 100% vars': 1.00
}

# Número de variables por especificación
n_vars_total = X_num_scaled.shape[1]
print(f"Total de variables disponibles: {n_vars_total}")
print()

In [ ]:
#Ahora guardamos los betas estimados y sus errores estandar en la base de entrenamiento
resultados_modelos = {}
betas_table = pd.DataFrame()

for spec_name, prop_vars in especificaciones.items():
    
    # Calcular número de variables a usar
    n_vars = max(1, int(np.ceil(n_vars_total * prop_vars)))
    
    # Seleccionar primeras n_vars variables
    X_train_subset = X_train[:, :n_vars]
    X_test_subset = X_test[:, :n_vars]
    
    # Entrenar modelo
    modelo = LinearRegression()
    modelo.fit(X_train_subset, y_train)
    
    # Predicciones
    y_pred_train = modelo.predict(X_train_subset)
    y_pred_test = modelo.predict(X_test_subset)
    
    # Métricas
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    
    # Calcular errores estándar de los betas
    # residuos
    residuos = y_train - y_pred_train
    sigma_sq = np.sum(residuos**2) / (len(y_train) - n_vars - 1)
    
    # Matriz de covarianza de betas
    X_train_with_const = np.column_stack([np.ones(X_train_subset.shape[0]), X_train_subset])
    XtX_inv = np.linalg.inv(X_train_with_const.T @ X_train_with_const)
    var_betas = sigma_sq * np.diag(XtX_inv)
    se_betas = np.sqrt(var_betas)
    
    # Guardar resultados
    resultados_modelos[spec_name] = {
        'n_vars': n_vars,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'mse_train': mse_train,
        'mse_test': mse_test,
        'rmse_test': rmse_test,
        'modelo': modelo,
        'betas': modelo.coef_,
        'intercept': modelo.intercept_,
        'se_betas': se_betas[1:]  # Excluir el SE del intercept
    }
    
    print(f"{spec_name} (n={n_vars} variables)")
    print(f"  R² Train: {r2_train:.4f} | R² Test: {r2_test:.4f}")
    print(f"  MSE Test: {mse_test:.2f} | RMSE Test: {rmse_test:.2f}")
    print()

Ahora visualizamos en una tabla los resultados y cómo cambian los $\beta$s estimados

In [ ]:
# Obtener nombres de variables
var_names = X_num.columns.tolist()

# Crear tabla
tabla_betas = pd.DataFrame()

# Primera fila: Intercepto
intercepts = [resultados_modelos[spec]['intercept'] for spec in especificaciones.keys()]
tabla_betas.loc['Intercept', list(especificaciones.keys())] = intercepts

# Agregar betas para cada variable (solo las que se usan en cada especificación)
for idx, var in enumerate(var_names):
    # Fila con betas
    betas_row = []
    ses_row = []
    
    for spec_name in especificaciones.keys():
        n_vars_spec = resultados_modelos[spec_name]['n_vars']
        
        if idx < n_vars_spec:
            beta_val = resultados_modelos[spec_name]['betas'][idx]
            se_val = resultados_modelos[spec_name]['se_betas'][idx]
            betas_row.append(f"{beta_val:.1f}")
            ses_row.append(f"({se_val:.1f})")
        else:
            betas_row.append("-")
            ses_row.append("-")
    
    # Agregar a tabla
    tabla_betas.loc[f'{var}', list(especificaciones.keys())] = betas_row
    tabla_betas.loc[f'', list(especificaciones.keys())] = ses_row  # Fila vacía con SE

# Agregar métricas finales
tabla_betas.loc['', list(especificaciones.keys())] = [''] * len(especificaciones)  # Espacio
tabla_betas.loc['R² (Train)', list(especificaciones.keys())] = [
    f"{resultados_modelos[spec]['r2_train']:.1f}" for spec in especificaciones.keys()
]
tabla_betas.loc['R² (Test)', list(especificaciones.keys())] = [
    f"{resultados_modelos[spec]['r2_test']:.1f}" for spec in especificaciones.keys()
]
tabla_betas.loc['MSE (Test)', list(especificaciones.keys())] = [
    f"{resultados_modelos[spec]['mse_test']:.1f}" for spec in especificaciones.keys()
]
tabla_betas.loc['RMSE (Test)', list(especificaciones.keys())] = [
    f"{resultados_modelos[spec]['rmse_test']:.1f}" for spec in especificaciones.keys()
]
tabla_betas.loc['N variables', list(especificaciones.keys())] = [
    f"{resultados_modelos[spec]['n_vars']}" for spec in especificaciones.keys()
]
